In [2]:
# # Data Analysis & Visualization
# ## International Green Power Utilities Financial Analysis

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ## 1. Load Data
df = pd.read_csv("green_power_data.csv")
print("Data loaded successfully")
print("Shape:", df.shape)

# ## 2. Clean Column Names
def clean_column_name(col):
    if '(' in str(col) and ')' in str(col):
        s = str(col).find('(') + 1
        e = str(col).find(')')
        return str(col)[s:e].strip()
    return str(col).strip()

df.columns = [clean_column_name(c) for c in df.columns]

# ## 3. Rename Variables
mapping = {
    'conm': 'Company',
    'fyear': 'Year',
    'at':   'Total_Assets',
    'lt':   'Total_Liabilities',
    'oiadp':'Operating_Income',
    'seq':  'Equity',
    'nicon':'Net_Income'
}
df = df.rename(columns={k:v for k,v in mapping.items() if k in df.columns})

# ## 4. Convert to Numeric
for col in ['Total_Assets','Total_Liabilities','Operating_Income','Equity','Net_Income']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# ## 5. Sort
df = df.sort_values(['Company','Year']).reset_index(drop=True)

# ## 6. Compute Financial Ratios
if 'Net_Income' in df and 'Equity' in df:
    df['ROE'] = df['Net_Income'] / df['Equity']

if 'Net_Income' in df and 'Operating_Income' in df:
    df['Net_Margin'] = df['Net_Income'] / df['Operating_Income']

if 'Total_Liabilities' in df and 'Total_Assets' in df:
    df['Debt_Ratio'] = df['Total_Liabilities'] / df['Total_Assets']

if 'Operating_Income' in df and 'Total_Assets' in df:
    df['Operating_Margin'] = df['Operating_Income'] / df['Total_Assets']

# ## 7. Growth Rates
for gcol in ['Operating_Income','Net_Income','Total_Assets']:
    if gcol in df:
        df[f'{gcol}_Growth'] = df.groupby('Company')[gcol].pct_change() * 100

df = df.replace([np.inf, -np.inf], np.nan)

# ## 8. Overview Table
print("\nCleaned Data Preview")
print(df.head())

# ## 9. Companies & Years
print("\nCompanies:", df['Company'].unique())
print("Year Range:", df['Year'].min(), "-", df['Year'].max())

# ## 10. Performance Dashboard (Same as Streamlit)
print("\n[Plot] Performance Dashboard")

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Operating Income Trend",
        "Profitability Ratios",
        "Debt Ratio Trend",
        "Growth Rate Comparison"
    ),
    vertical_spacing=0.14
)

for comp in df['Company'].unique():
    sub = df[df['Company'] == comp]

    # Top-left: Operating Income
    fig.add_trace(go.Scatter(
        x=sub['Year'], y=sub['Operating_Income'],
        name=f"Operating Income {comp}", mode="lines+markers"
    ), row=1, col=1)

    # Top-right: ROE
    if 'ROE' in sub:
        fig.add_trace(go.Scatter(
            x=sub['Year'], y=sub['ROE']*100,
            name=f"ROE {comp}", mode="lines+markers"
        ), row=1, col=2)

    # Bottom-left: Debt Ratio
    if 'Debt_Ratio' in sub:
        fig.add_trace(go.Scatter(
            x=sub['Year'], y=sub['Debt_Ratio']*100,
            name=f"Debt Ratio {comp}", mode="lines+markers"
        ), row=2, col=1)

    # Bottom-right: Growth
    if 'Operating_Income_Growth' in sub:
        fig.add_trace(go.Bar(
            x=sub['Year'], y=sub['Operating_Income_Growth'],
            name=f"OpInc Growth {comp}"
        ), row=2, col=2)

fig.update_layout(height=750, title_text="Financial Performance Dashboard", showlegend=True)
fig.show()

# ## 11. Latest Year Company Comparison
print("\n[Plot] Latest Year Comparison")

latest_y = df['Year'].max()
latest = df[df['Year'] == latest_y]

fig2 = px.bar(
    latest,
    x="Company",
    y="Operating_Income",
    color="Company",
    title=f"Operating Income in {latest_y}"
)
fig2.update_layout(height=450)
fig2.show()

# ## 12. Capital Structure Trend
print("\n[Plot] Debt Ratio Trend")

fig3 = go.Figure()
for comp in df['Company'].unique():
    sub = df[df['Company'] == comp]
    if 'Debt_Ratio' in sub:
        fig3.add_trace(go.Scatter(
            x=sub['Year'], y=sub['Debt_Ratio']*100,
            name=f"{comp}", mode="lines+markers"
        ))

fig3.update_layout(
    title="Debt Ratio Over Time",
    yaxis_title="Debt Ratio (%)",
    height=450
)
fig3.show()

# ## 13. Done
print(" Analysis and all plots generated successfully.")

Data loaded successfully
Shape: (28, 14)

Cleaned Data Preview
   fic costat   datafmt indfmt consol   gvkey  datadate   Company  Year  \
0  ITA      A  HIST_STD   INDL      C  201794  31/12/18  ENEL SPA  2018   
1  ITA      A  HIST_STD   INDL      C  201794  31/12/19  ENEL SPA  2019   
2  ITA      A  HIST_STD   INDL      C  201794  31/12/20  ENEL SPA  2020   
3  ITA      A  HIST_STD   INDL      C  201794  31/12/21  ENEL SPA  2021   
4  ITA      A  HIST_STD   INDL      C  201794  31/12/22  ENEL SPA  2022   

   Total_Assets  ...  Operating_Income   Equity  Net_Income       ROE  \
0      165424.0  ...            9371.0  31720.0      4789.0  0.150977   
1      171426.0  ...           10530.0  30377.0      2174.0  0.071567   
2      163453.0  ...           10959.0  28325.0      2610.0  0.092145   
3      206940.0  ...            8156.0  29653.0      3189.0  0.107544   
4      219618.0  ...           10436.0  28657.0      3514.0  0.122623   

   Net_Margin  Debt_Ratio  Operating_Margin  Op


[Plot] Latest Year Comparison



[Plot] Debt Ratio Trend


 Analysis and all plots generated successfully.
